# ARC Agent Visualization Notebook

This notebook provides an end-to-end offline workflow for ARC agent experiments on Kaggle:
- install local wheel dependencies,
- write a custom agent,
- run the official main.py pipeline in offline mode,
- collect recordings and score summaries,
- replay frames in an interactive UI.

## Notes on provenance and motivation
- Cells 1 and 2 were copied and adapted from the official sample notebook:
  [ARC3 Sample Submission - Random Agent](https://www.kaggle.com/code/inversion/arc3-sample-submission-random-agent)
- The sample notebook is mainly focused on copying an agent into templates and producing a valid submission artifact.
- This notebook extends that baseline with additional tooling to:
  - run the copied agent offline,
  - compute action-count and score statistics on public games,
  - visualize recordings to inspect behavior and debug the agent more effectively.
- Most importantly, this workflow executes the agent through the real main.py command path, so behavior stays as close as possible to the official evaluation runtime, instead of relying on a custom simulator built from the provided environment files.

## Cell-by-cell overview
1. Dependency setup: installs required packages from local wheel files.
2. Agent definition: writes my_agent.py with a random baseline agent.
3. Offline runner and dashboard: defines helper utilities and run_arc(...) to execute runs, parse logs, and save summaries.
4. Run execution: launches a baseline run and stores the result dictionary.
5. Recording visualization UI: loads saved JSONL recordings and provides interactive playback controls.
6. Scratch cell: reserved for ad-hoc analysis after the main pipeline runs.

## Cell 1 - Dependency Installation

Installs ARC runtime dependencies from local wheel files in the Kaggle input dataset.

Why this cell exists:
- avoids internet access requirements,
- keeps package versions aligned with competition-provided artifacts,
- prepares the environment for all later cells.

In [1]:
# Cell 1: Install required dependencies from local ARC wheel files.
# This keeps the notebook self-contained in restricted Kaggle environments.
!python -m pip install -q --no-index --find-links=/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.


## Cell 2 - Agent Definition

Writes a baseline random agent to /kaggle/working/my_agent.py.

What this cell provides:
- a minimal but valid policy implementation,
- deterministic-ish seeding per game run,
- readable reasoning fields for debugging and visualization.

In [2]:
%%writefile /kaggle/working/my_agent.py
# Cell 2: Define a baseline agent and write it to disk so the ARC runner can import it.

import random
import time
from typing import Any

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


class MyAgent(Agent):
    """Random baseline agent used for offline testing and visualization.

    Behavior summary:
    - Resets the game when the current state requires a fresh start.
    - Otherwise picks a random non-reset action.
    - For complex actions, randomly samples coordinates in the 64x64 range.

    This class is intentionally simple and acts as a template for future policies.
    """

    # A finite cap avoids unbounded trajectories during debugging.
    MAX_ACTIONS = 500

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        """Initialize the agent and create a game-specific random seed.

        The seed combines current time and the game identifier hash so that
        consecutive runs are varied while still being deterministic within a run.
        """
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        random.seed(seed)

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        """Stop only when the environment reports a win state."""
        return latest_frame.state is GameState.WIN

    def _set_reasoning(self, action: GameAction, reasoning) -> None:
        """Attach reasoning to both action surfaces when available.

        Some downstream tooling reads `action.reasoning`, while other code paths
        inspect `action.action_data.reasoning`. This helper keeps both in sync.
        """
        action.reasoning = reasoning
        try:
            action.action_data.reasoning = reasoning
        except Exception:
            # Some actions may not expose action_data in the expected shape.
            pass

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        """Choose the next action using a pure random policy.

        Policy details:
        - If the game is not started or already over, return RESET.
        - Otherwise, sample uniformly from non-RESET actions.
        - Add human-readable reasoning for easier debugging.
        """
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            action = GameAction.RESET
            self._set_reasoning(action, "Reset because the game is not started or already over.")
            return action

        action = random.choice([a for a in GameAction if a is not GameAction.RESET])

        if action.is_simple():
            self._set_reasoning(action, f"Random policy selected action: {action.value}")
        elif action.is_complex():
            action.set_data(
                {
                    "x": random.randint(0, 63),
                    "y": random.randint(0, 63),
                }
            )
            self._set_reasoning(
                action,
                {
                    "desired_action": f"{action.value}",
                    "policy": "random",
                    "note": "Coordinates were sampled uniformly in [0, 63].",
                },
            )

        return action

Writing /kaggle/working/my_agent.py


## Cell 3 - Offline Runner, Parsers, and Live Dashboard

Defines the execution engine used to run ARC offline while preserving the official repository workflow.

Main responsibilities:
- copy and patch a writable repo clone,
- create an offline .env and local /api/games stub,
- stream and parse logs into structured status,
- render a live dashboard during execution,
- export summary and score artifacts.

In [3]:
# Cell 3: Define offline ARC execution helpers, log parsers, and a live status dashboard.

import os
import re
import sys
import time
import json
import html
import socket
import shutil
import textwrap
import subprocess
from pathlib import Path
from datetime import datetime

import requests
import ipywidgets as widgets
from IPython.display import display


def _find_free_port(host: str = "127.0.0.1", start_port: int = 8765, max_tries: int = 100) -> int:
    """Return the first available TCP port on the given host.

    The function tries ports in ascending order from `start_port` and stops
    after `max_tries` attempts.
    """
    port = start_port
    for _ in range(max_tries):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind((host, port))
                return port
            except OSError:
                port += 1
    raise RuntimeError("Could not find a free local port.")


def _esc(x: object) -> str:
    """Escape arbitrary content for safe HTML rendering in widgets."""
    return html.escape(str(x))


def _slugify(text: str | None) -> str:
    """Convert free text into a filesystem-friendly slug.

    Keeps lowercase alphanumeric characters and uses `-` as a separator.
    """
    if not text:
        return ""
    text = text.strip().lower()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    text = re.sub(r"-+", "-", text).strip("-")
    return text


def _make_run_dir(base_dir: Path, description: str | None = None) -> Path:
    """Create a unique run directory under `base_dir`.

    Directory format:
    - runs-YYYYMMDD-HHMMSS
    - runs-YYYYMMDD-HHMMSS-description

    If a directory with the same name exists, a numeric suffix is appended.
    """
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    slug = _slugify(description)
    name = f"runs-{timestamp}" + (f"-{slug}" if slug else "")
    run_dir = base_dir / name

    if not run_dir.exists():
        run_dir.mkdir(parents=True, exist_ok=False)
        return run_dir

    for i in range(1, 1000):
        candidate = base_dir / f"{name}-{i:02d}"
        if not candidate.exists():
            candidate.mkdir(parents=True, exist_ok=False)
            return candidate

    raise RuntimeError("Could not allocate a unique run directory.")


def _strip_log_prefix(line: str) -> str:
    """Remove timestamp/level prefixes from log lines before parsing."""
    return re.sub(r"^\d{4}-\d{2}-\d{2}.*?\|\s*INFO\s*\|\s*", "", line)


def _extract_scorecard_json(final_lines: list[str]) -> dict | None:
    """Extract and parse the final JSON scorecard from textual logs.

    The function starts from the first `{` and tracks brace balance until
    the JSON object is complete.
    """
    started = False
    brace_balance = 0
    chunks = []

    for raw in final_lines:
        line = raw.rstrip("\n")
        stripped = _strip_log_prefix(line)

        if not started:
            if "{" in stripped:
                idx = stripped.find("{")
                piece = stripped[idx:]
                chunks.append(piece)
                brace_balance += piece.count("{") - piece.count("}")
                started = True
                if brace_balance == 0:
                    break
        else:
            chunks.append(stripped)
            brace_balance += stripped.count("{") - stripped.count("}")
            if brace_balance == 0:
                break

    if not chunks:
        return None

    try:
        return json.loads("\n".join(chunks))
    except Exception:
        return None


def _summary_rows_from_status(game_order: list[str], status: dict[str, dict]) -> list[dict]:
    """Convert internal status state into tabular summary rows."""
    rows = []
    for g in game_order:
        s = status[g]
        rows.append(
            {
                "game": g,
                "levels_completed": s.get("levels_completed", 0),
                "action_count": s.get("action_count", 0),
                "last_action": s.get("last_action", "-"),
                "fps": round(float(s.get("fps", 0.0)), 2),
                "score": round(float(s.get("score", 0.0)), 6) if s.get("score") is not None else None,
                "state": s.get("state", "waiting"),
            }
        )
    return rows


def _write_summary_csv(path: Path, rows: list[dict]) -> None:
    """Persist summary rows to a CSV file."""
    import csv

    fieldnames = ["game", "levels_completed", "action_count", "last_action", "fps", "score", "state"]
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def _write_summary_txt(path: Path, rows: list[dict], overall_score: float | None) -> None:
    """Write a human-readable plain text summary table."""
    headers = ["game", "levels", "actions", "last_action", "fps", "score", "state"]
    widths = {
        "game": 8,
        "levels": 8,
        "actions": 8,
        "last_action": 12,
        "fps": 8,
        "score": 10,
        "state": 10,
    }

    lines = []
    lines.append(
        f"{headers[0]:<{widths['game']}} "
        f"{headers[1]:>{widths['levels']}} "
        f"{headers[2]:>{widths['actions']}} "
        f"{headers[3]:<{widths['last_action']}} "
        f"{headers[4]:>{widths['fps']}} "
        f"{headers[5]:>{widths['score']}} "
        f"{headers[6]:<{widths['state']}}"
    )
    lines.append("-" * (sum(widths.values()) + 6))

    for r in rows:
        score_value = "-" if r["score"] is None else f"{r['score']:.6f}"
        lines.append(
            f"{str(r['game']):<{widths['game']}} "
            f"{str(r['levels_completed']):>{widths['levels']}} "
            f"{str(r['action_count']):>{widths['actions']}} "
            f"{str(r['last_action']):<{widths['last_action']}} "
            f"{str(r['fps']):>{widths['fps']}} "
            f"{score_value:>{widths['score']}} "
            f"{str(r['state']):<{widths['state']}}"
        )

    lines.append("")
    lines.append(f"Overall score: {overall_score:.6f}" if overall_score is not None else "Overall score: N/A")
    path.write_text("\n".join(lines), encoding="utf-8")


def run_arc(
    agent_src: str,
    agent_class_name: str,
    agent_cli_name: str,
    run_game: str = "all",
    *,
    description: str | None = None,
    input_root: str = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3",
    working_root: str = "/kaggle/working",
    repo_copy_name: str = "ARC-AGI-3-Agents",
    host: str = "127.0.0.1",
    port: int | None = 8765,
    overwrite_repo: bool = True,
    update_interval: float = 0.03,
):
    """Run ARC `main.py` in offline mode without modifying competition source files.

    Args:
        agent_src: Path to the custom agent file to copy into the repo templates.
        agent_class_name: Class name to expose in `agents/__init__.py`.
        agent_cli_name: CLI key used by `main.py --agent`.
        run_game:
            - "all": run all games in `environment_files`.
            - "<game_id>": run a single game.
        description: Optional suffix used in run directory naming.
        input_root: Read-only competition input root.
        working_root: Writable Kaggle working directory.
        repo_copy_name: Folder name for the writable repo clone.
        host: Host for the local stub API server.
        port: Port for the local stub API server. If None, auto-select one.
        overwrite_repo: Re-copy the repo before each run if True.
        update_interval: Footer refresh cadence in seconds during streaming logs.

    Returns:
        A dictionary containing exit code, score, run paths, and metadata.

    Artifacts created per run:
        runs-{timestamp}-{description}/
            recordings/
            summary.csv
            summary.txt
            scorecard.json
            <agent_copy>.py
            run.log
    """
    if run_game is None:
        run_game = "all"

    input_root = Path(input_root)
    working_root = Path(working_root)

    env_dir = input_root / "environment_files"
    repo_src = input_root / "ARC-AGI-3-Agents"
    repo_dst = working_root / repo_copy_name
    agent_src = Path(agent_src)

    if not env_dir.exists():
        raise FileNotFoundError(f"ENVIRONMENTS_DIR not found: {env_dir}")
    if not repo_src.exists():
        raise FileNotFoundError(f"Repo not found: {repo_src}")
    if not agent_src.exists():
        raise FileNotFoundError(f"Agent source not found: {agent_src}")

    if port is None:
        port = _find_free_port(host=host)

    root_url = f"http://{host}:{port}"

    cmd = [sys.executable, "main.py", "--agent", agent_cli_name]
    if run_game != "all":
        cmd += ["--game", run_game]

    run_dir = _make_run_dir(working_root, description=description)
    recordings_dir = run_dir / "recordings"
    recordings_dir.mkdir(parents=True, exist_ok=True)

    # Save an immutable copy of the exact agent used in this run.
    agent_copy_path = run_dir / agent_src.name
    shutil.copy(agent_src, agent_copy_path)

    raw_log_path = run_dir / "run.log"
    summary_csv_path = run_dir / "summary.csv"
    summary_txt_path = run_dir / "summary.txt"
    scorecard_json_path = run_dir / "scorecard.json"

    if overwrite_repo and repo_dst.exists():
        shutil.rmtree(repo_dst)

    if not repo_dst.exists():
        shutil.copytree(repo_src, repo_dst)

    # Copy custom agent into the templates package so `main.py` can import it.
    agent_module_name = agent_src.stem
    dst_agent_path = repo_dst / "agents" / "templates" / f"{agent_module_name}.py"
    shutil.copy(agent_src, dst_agent_path)

    # Replace eager imports with a minimal `agents/__init__.py` that exposes only needed classes.
    agents_init = textwrap.dedent(f"""
    from typing import Type
    from dotenv import load_dotenv

    from .agent import Agent, Playback
    from .swarm import Swarm
    from .templates.random_agent import Random
    from .templates.{agent_module_name} import {agent_class_name}

    load_dotenv()

    AVAILABLE_AGENTS: dict[str, Type[Agent]] = {{
        "random": Random,
        "{agent_cli_name}": {agent_class_name},
    }}
    """).strip() + "\n"
    (repo_dst / "agents" / "__init__.py").write_text(agents_init, encoding="utf-8")

    # Create a local offline `.env` that points to the copied environments and recordings directory.
    env_text = textwrap.dedent(f"""
    OPERATION_MODE=OFFLINE
    ENVIRONMENTS_DIR={env_dir}
    RECORDINGS_DIR={recordings_dir}

    SCHEME=http
    HOST={host}
    PORT={port}
    ARC_BASE_URL={root_url}
    ARC_API_KEY=offline
    """).strip() + "\n"
    (repo_dst / ".env").write_text(env_text, encoding="utf-8")

    # Provide a minimal `/api/games` endpoint expected by the original `main.py` workflow.
    fake_api_code = textwrap.dedent(f"""
    import json
    from pathlib import Path
    from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

    ENV_DIR = Path(r"{env_dir}")

    def list_games():
        if not ENV_DIR.exists():
            return []
        games = []
        for p in sorted(ENV_DIR.iterdir()):
            if p.is_dir():
                games.append({{"game_id": p.name}})
        return games

    class Handler(BaseHTTPRequestHandler):
        def do_GET(self):
            if self.path == "/api/games":
                payload = json.dumps(list_games()).encode("utf-8")
                self.send_response(200)
                self.send_header("Content-Type", "application/json")
                self.send_header("Content-Length", str(len(payload)))
                self.end_headers()
                self.wfile.write(payload)
                return

            self.send_response(404)
            self.end_headers()

        def log_message(self, format, *args):
            return

    if __name__ == "__main__":
        server = ThreadingHTTPServer(("{host}", {port}), Handler)
        server.serve_forever()
    """).strip() + "\n"
    fake_api_path = repo_dst / "fake_games_api.py"
    fake_api_path.write_text(fake_api_code, encoding="utf-8")

    # Precompile regexes used to parse streaming logs.
    re_game_list = re.compile(r"Game list:\s*\[(.*)\]")
    re_action = re.compile(
        r"\b([a-z0-9]+)\s*-\s*(ACTION\d+|RESET)\:\s*count\s*(\d+),\s*levels completed\s*(\d+),\s*avg fps\s*([0-9.]+)"
    )
    re_recording_done = re.compile(r"recording for ([a-z0-9]+)\.[^. ]+ is available")
    re_scorecard_start = re.compile(r"--- FINAL SCORECARD REPORT ---")

    # Build initial dashboard state from available environments.
    initial_games = sorted([p.name for p in env_dir.iterdir() if p.is_dir()])
    if run_game != "all":
        initial_games = [run_game] if run_game in initial_games else [run_game]

    status = {
        g: {
            "game": g,
            "levels_completed": 0,
            "action_count": 0,
            "last_action": "-",
            "fps": 0.0,
            "score": None,
            "state": "waiting",
            "done": False,
        }
        for g in initial_games
    }
    game_order = initial_games[:]

    # ---------------------------
    # Dashboard widgets and styling
    # ---------------------------
    monospace = "ui-monospace, SFMono-Regular, Menlo, Consolas, monospace"

    title_html = widgets.HTML(
        value="",
        layout=widgets.Layout(width="100%", margin="0 0 6px 0")
    )

    header_bg = "#111827"
    header_fg = "#f9fafb"
    header_border = "#0b1220"

    header_style = (
        "font-family:{font};font-size:12px;font-weight:700;"
        "padding:4px 8px;"
        f"background:{header_bg};color:{header_fg};"
        f"border-bottom:1px solid {header_border};"
    )

    cell_style = (
        "font-family:{font};font-size:12px;"
        "padding:2px 8px;"
        "border-bottom:1px solid #e5e7eb;"
        "white-space:nowrap;overflow:hidden;text-overflow:ellipsis;"
        "line-height:1.2;"
    )

    def make_header_cell(text: str, width: str) -> widgets.HTML:
        """Create one header cell for the dashboard table."""
        return widgets.HTML(
            value=f"<div style='{header_style.format(font=monospace)}'>{_esc(text)}</div>",
            layout=widgets.Layout(width=width)
        )

    def make_cell(text: str = "", width: str = "120px", align: str = "left") -> widgets.HTML:
        """Create one body cell for the dashboard table."""
        return widgets.HTML(
            value=f"<div style='{cell_style.format(font=monospace)}text-align:{align};'>{_esc(text)}</div>",
            layout=widgets.Layout(width=width)
        )

    header = widgets.HBox(
        [
            make_header_cell("game", "88px"),
            make_header_cell("levels", "68px"),
            make_header_cell("actions", "72px"),
            make_header_cell("last_action", "96px"),
            make_header_cell("fps", "64px"),
            make_header_cell("score", "78px"),
            make_header_cell("state", "80px"),
        ],
        layout=widgets.Layout(width="100%")
    )

    row_widgets = {}
    row_boxes = []
    row_render_cache = {}
    title_cache = {"value": None}
    footer_cache = {"value": None}

    def state_html_text(state: str) -> str:
        """Render the state text with semantic colors."""
        color = {
            "waiting": "#6b7280",
            "loaded": "#2563eb",
            "running": "#d97706",
            "done": "#16a34a",
        }.get(state, "#374151")
        return (
            f"<div style='{cell_style.format(font=monospace)}"
            f"color:{color};font-weight:700'>{_esc(state)}</div>"
        )

    def score_text(score: float | None) -> str:
        """Format a score value for compact table display."""
        if score is None:
            return "-"
        return f"{float(score):.4f}"

    for g in game_order:
        w_game = make_cell(g, "88px")
        w_levels = make_cell("0", "68px", "right")
        w_actions = make_cell("0", "72px", "right")
        w_action = make_cell("-", "96px")
        w_fps = make_cell("0.00", "64px", "right")
        w_score = make_cell("-", "78px", "right")
        w_state = widgets.HTML(
            value=state_html_text("waiting"),
            layout=widgets.Layout(width="80px")
        )

        row_widgets[g] = {
            "game": w_game,
            "levels": w_levels,
            "actions": w_actions,
            "last_action": w_action,
            "fps": w_fps,
            "score": w_score,
            "state": w_state,
        }
        row_render_cache[g] = {}

        row_boxes.append(
            widgets.HBox(
                [w_game, w_levels, w_actions, w_action, w_fps, w_score, w_state],
                layout=widgets.Layout(width="100%")
            )
        )

    table_box = widgets.VBox(
        [header] + row_boxes,
        layout=widgets.Layout(
            width="100%",
            border="1px solid #d1d5db",
            overflow="hidden",
        )
    )

    footer_html = widgets.HTML(
        value="",
        layout=widgets.Layout(width="100%", margin="6px 0 0 0")
    )

    dashboard = widgets.VBox(
        [title_html, table_box, footer_html],
        layout=widgets.Layout(width="100%")
    )
    display(dashboard)

    def set_widget_html(widget: widgets.Widget, new_value: str, cache_dict: dict, key: str) -> None:
        """Update widget HTML only when value changes to reduce redraw overhead."""
        if cache_dict.get(key) != new_value:
            widget.value = new_value
            cache_dict[key] = new_value

    def compute_overall_score() -> float | None:
        """Compute mean score across games that already have score values."""
        scores = [s["score"] for s in status.values() if s.get("score") is not None]
        if not scores:
            return None
        return sum(scores) / len(scores)

    def update_title() -> None:
        """Refresh dashboard header block with run metadata."""
        value = (
            f"<div style='font-family:{monospace};font-size:12px;line-height:1.35'>"
            f"<b>Running:</b> {_esc(' '.join(cmd))}<br>"
            f"<b>Run dir:</b> {_esc(run_dir)}<br>"
            f"<b>Recordings:</b> {_esc(recordings_dir)}<br>"
            f"<b>Stub API:</b> {_esc(root_url)}/api/games"
            f"</div>"
        )
        if title_cache["value"] != value:
            title_html.value = value
            title_cache["value"] = value

    def update_footer() -> None:
        """Refresh dashboard footer with finished/running counters and overall score."""
        done_count = sum(1 for g in game_order if status[g]["done"])
        running_count = sum(1 for g in game_order if status[g]["state"] == "running")
        overall = compute_overall_score()
        overall_part = f" &nbsp;&nbsp; <b>Overall score:</b> {overall:.6f}" if overall is not None else ""
        value = (
            f"<div style='font-family:{monospace};font-size:12px;line-height:1.35'>"
            f"<b>Finished:</b> {done_count}/{len(game_order)} &nbsp;&nbsp; "
            f"<b>Running:</b> {running_count}"
            f"{overall_part}"
            f"</div>"
        )
        if footer_cache["value"] != value:
            footer_html.value = value
            footer_cache["value"] = value

    def update_row(game: str) -> None:
        """Refresh one table row from the current game status dictionary."""
        s = status[game]
        rw = row_widgets[game]
        cache = row_render_cache[game]

        html_levels = f"<div style='{cell_style.format(font=monospace)}text-align:right'>{s['levels_completed']}</div>"
        html_actions = f"<div style='{cell_style.format(font=monospace)}text-align:right'>{s['action_count']}</div>"
        html_action = f"<div style='{cell_style.format(font=monospace)}'>{_esc(s['last_action'])}</div>"
        html_fps = f"<div style='{cell_style.format(font=monospace)}text-align:right'>{s['fps']:.2f}</div>"
        html_score = f"<div style='{cell_style.format(font=monospace)}text-align:right'>{_esc(score_text(s['score']))}</div>"
        html_state = state_html_text(s["state"])

        set_widget_html(rw["levels"], html_levels, cache, "levels")
        set_widget_html(rw["actions"], html_actions, cache, "actions")
        set_widget_html(rw["last_action"], html_action, cache, "last_action")
        set_widget_html(rw["fps"], html_fps, cache, "fps")
        set_widget_html(rw["score"], html_score, cache, "score")
        set_widget_html(rw["state"], html_state, cache, "state")

    def ensure_game(game: str) -> None:
        """Ensure dashboard data structures contain a row for `game`.

        This supports late game discovery from logs when game list parsing occurs
        after the dashboard has already been rendered.
        """
        if game in status:
            return

        status[game] = {
            "game": game,
            "levels_completed": 0,
            "action_count": 0,
            "last_action": "-",
            "fps": 0.0,
            "score": None,
            "state": "waiting",
            "done": False,
        }

        w_game = make_cell(game, "88px")
        w_levels = make_cell("0", "68px", "right")
        w_actions = make_cell("0", "72px", "right")
        w_action = make_cell("-", "96px")
        w_fps = make_cell("0.00", "64px", "right")
        w_score = make_cell("-", "78px", "right")
        w_state = widgets.HTML(
            value=state_html_text("waiting"),
            layout=widgets.Layout(width="80px")
        )

        row_widgets[game] = {
            "game": w_game,
            "levels": w_levels,
            "actions": w_actions,
            "last_action": w_action,
            "fps": w_fps,
            "score": w_score,
            "state": w_state,
        }
        row_render_cache[game] = {}

        new_row = widgets.HBox(
            [w_game, w_levels, w_actions, w_action, w_fps, w_score, w_state],
            layout=widgets.Layout(width="100%")
        )

        table_box.children = tuple(list(table_box.children) + [new_row])
        game_order.append(game)
        update_row(game)

    update_title()
    update_footer()

    server_proc = None
    proc = None
    scorecard_dict = None

    with raw_log_path.open("w", encoding="utf-8") as raw_log_file:
        try:
            # Start a local API shim that serves `/api/games` expected by `main.py`.
            server_proc = subprocess.Popen(
                [sys.executable, "fake_games_api.py"],
                cwd=repo_dst,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.STDOUT,
            )

            # Wait until the local shim responds before launching the agent run.
            ready = False
            for _ in range(60):
                try:
                    r = requests.get(f"{root_url}/api/games", timeout=0.5)
                    if r.ok:
                        ready = True
                        break
                except Exception:
                    time.sleep(0.1)

            if not ready:
                raise RuntimeError("Local stub API did not start in time.")

            env = os.environ.copy()
            env["MPLBACKEND"] = "agg"
            env["PYTHONUNBUFFERED"] = "1"

            # Launch the official pipeline and stream logs line by line.
            proc = subprocess.Popen(
                cmd,
                cwd=repo_dst,
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )

            last_footer_update = 0.0
            final_lines = []
            raw_tail = []
            in_final_summary = False

            for line in proc.stdout:
                raw_log_file.write(line)
                raw_log_file.flush()

                line = line.rstrip("\n")
                raw_tail.append(line)
                if len(raw_tail) > 400:
                    raw_tail.pop(0)

                # Once final scorecard section starts, collect lines for JSON extraction.
                if in_final_summary:
                    final_lines.append(line)
                    continue

                if re_scorecard_start.search(line):
                    in_final_summary = True
                    final_lines.append(line)
                    continue

                # Update known games from `Game list: [...]` logs.
                m = re_game_list.search(line)
                if m:
                    try:
                        parsed = "[" + m.group(1).replace("'", '"') + "]"
                        games = json.loads(parsed)
                        for g in games:
                            ensure_game(g)
                            if status[g]["state"] == "waiting":
                                status[g]["state"] = "loaded"
                                update_row(g)
                        update_footer()
                    except Exception:
                        pass
                    continue

                # Parse per-action progress logs and refresh one row.
                m = re_action.search(line)
                if m:
                    game, action_name, count, levels_completed, fps = m.groups()
                    ensure_game(game)
                    status[game]["last_action"] = action_name
                    status[game]["action_count"] = int(count) + 1
                    status[game]["levels_completed"] = int(levels_completed)
                    status[game]["fps"] = float(fps)
                    status[game]["state"] = "running"
                    update_row(game)

                    now = time.time()
                    if now - last_footer_update >= update_interval:
                        update_footer()
                        last_footer_update = now
                    continue

                # Mark game as done when recording availability is announced.
                m = re_recording_done.search(line)
                if m:
                    game = m.group(1)
                    ensure_game(game)
                    status[game]["done"] = True
                    status[game]["state"] = "done"
                    update_row(game)
                    update_footer()
                    continue

            exit_code = proc.wait()

            # Parse and persist final scorecard if present.
            scorecard_dict = _extract_scorecard_json(final_lines)
            if scorecard_dict is not None:
                scorecard_json_path.write_text(
                    json.dumps(scorecard_dict, ensure_ascii=False, indent=2),
                    encoding="utf-8",
                )

                for env_entry in scorecard_dict.get("environments", []):
                    env_id = env_entry.get("id", "")
                    game = env_id.split("-", 1)[0] if env_id else None
                    if not game:
                        continue
                    ensure_game(game)

                    status[game]["score"] = float(env_entry.get("score", 0.0))
                    status[game]["levels_completed"] = int(env_entry.get("levels_completed", status[game]["levels_completed"]))
                    status[game]["action_count"] = int(env_entry.get("actions", status[game]["action_count"]))
                    status[game]["done"] = True
                    status[game]["state"] = "done"
                    update_row(game)

            update_footer()

            rows = _summary_rows_from_status(game_order, status)
            overall_score = compute_overall_score()

            _write_summary_csv(summary_csv_path, rows)
            _write_summary_txt(summary_txt_path, rows, overall_score)

            print()
            print("=" * 80)
            print(f"Overall score: {overall_score:.6f}" if overall_score is not None else "Overall score: N/A")
            print("=" * 80)
            print(f"Run dir: {run_dir}")

            result = {
                "exit_code": exit_code,
                "overall_score": overall_score,
                "run_dir": str(run_dir),
                "recordings_dir": str(recordings_dir),
                "summary_csv": str(summary_csv_path),
                "summary_txt": str(summary_txt_path),
                "scorecard_json": str(scorecard_json_path) if scorecard_json_path.exists() else None,
                "agent_copy": str(agent_copy_path),
                "log_path": str(raw_log_path),
                "repo_dst": str(repo_dst),
                "env_dir": str(env_dir),
                "root_url": root_url,
                "agent_module_name": agent_module_name,
                "agent_class_name": agent_class_name,
                "agent_cli_name": agent_cli_name,
                "run_game": run_game,
                "description": description,
            }

            print()
            print(result)
            return result

        finally:
            # Make sure both child processes are stopped even on errors.
            if proc is not None and proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(timeout=3)
                except subprocess.TimeoutExpired:
                    proc.kill()

            if server_proc is not None and server_proc.poll() is None:
                server_proc.terminate()
                try:
                    server_proc.wait(timeout=3)
                except subprocess.TimeoutExpired:
                    server_proc.kill()

## Cell 4 - Run the Experiment

Calls run_arc(...) with the custom agent and executes the selected game scope.

Outputs include:
- terminal progress and final score,
- paths to generated artifacts,
- a result dictionary saved in the result variable.

In [4]:
# Cell 4: Execute one offline run using the custom agent and capture output metadata.
result = run_arc(
    agent_src="/kaggle/working/my_agent.py",
    agent_class_name="MyAgent",
    agent_cli_name="myagent",
    run_game="all",
    description="random-baseline-v1",
)


Overall score: 0.005710
Run dir: /kaggle/working/runs-20260403-110444-random-baseline-v1

{'exit_code': 0, 'overall_score': 0.00571007402055847, 'run_dir': '/kaggle/working/runs-20260403-110444-random-baseline-v1', 'recordings_dir': '/kaggle/working/runs-20260403-110444-random-baseline-v1/recordings', 'summary_csv': '/kaggle/working/runs-20260403-110444-random-baseline-v1/summary.csv', 'summary_txt': '/kaggle/working/runs-20260403-110444-random-baseline-v1/summary.txt', 'scorecard_json': '/kaggle/working/runs-20260403-110444-random-baseline-v1/scorecard.json', 'agent_copy': '/kaggle/working/runs-20260403-110444-random-baseline-v1/my_agent.py', 'log_path': '/kaggle/working/runs-20260403-110444-random-baseline-v1/run.log', 'repo_dst': '/kaggle/working/ARC-AGI-3-Agents', 'env_dir': '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files', 'root_url': 'http://127.0.0.1:8765', 'agent_module_name': 'my_agent', 'agent_class_name': 'MyAgent', 'agent_cli_name': 'myagent', 'run_g

## Cell 5 - Interactive Recording Viewer

Builds an interactive UI to inspect generated recording frames.

Viewer capabilities:
- choose run and game,
- load JSONL frames on demand,
- scrub frame-by-frame with slider or autoplay,
- display board image and progress metadata.

In [5]:
# Cell 5: Build an interactive recording viewer for ARC run artifacts.

import json
import io
import base64
from pathlib import Path

import numpy as np
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, HTML

# Root directory where run artifacts are expected in Kaggle.
WORKING_DIR = Path("/kaggle/working")

# Official ARC color palette (index -> hex color).
ARC_COLORS_HEX = [
    "#FFFFFF", "#CCCCCC", "#999999", "#666666", "#333333", "#000000",
    "#E53AA3", "#FF7BCC", "#F93C31", "#1E93FF", "#88D8F1", "#FFDC00",
    "#FF851B", "#921231", "#4FCC30", "#A356D6",
]

# Playback and rendering constants.
FPS = 2
CANVAS_SIZE = 640
DISPLAY_MAX_WIDTH = 560


def hex_to_rgb(hex_code: str) -> tuple[int, int, int]:
    """Convert a `#RRGGBB` color string into an RGB tuple."""
    hex_code = hex_code.lstrip("#")
    return tuple(int(hex_code[i : i + 2], 16) for i in (0, 2, 4))


# Precompute the palette as a NumPy array for fast indexed color lookup.
ARC_COLORS_RGB = np.array([hex_to_rgb(c) for c in ARC_COLORS_HEX], dtype=np.uint8)


def safe_json_loads(line: str):
    """Parse a JSON line safely.

    Returns None for invalid JSON instead of raising, so malformed rows can be skipped.
    """
    try:
        return json.loads(line)
    except Exception:
        return None


def normalize_frame_to_single_grid(frame):
    """Normalize frame payloads to one 2D grid.

    Conversion rules (kept compatible with the original notebook behavior):
    - If input is 2D, use it directly.
    - If input is 3D, merge layers into a single 2D grid using:
      `flat[layer >= 0] = layer[layer >= 0]`
    - Return None when conversion is not possible.
    """
    if frame is None:
        return None

    try:
        grid = np.array(frame).astype(int)
    except Exception:
        return None

    if grid.size == 0:
        return None

    if grid.ndim == 3:
        flat = np.zeros((grid.shape[1], grid.shape[2]), dtype=int)
        for layer in grid:
            mask = layer >= 0
            flat[mask] = layer[mask]
        grid = flat
    else:
        grid = np.squeeze(grid)

    if grid.ndim != 2:
        return None

    return grid.tolist()


def load_recording_jsonl(jsonl_path: Path) -> list[dict]:
    """Load one recording JSONL file into a list of normalized frame dictionaries."""
    frames = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for step_idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue

            obj = safe_json_loads(line)
            if obj is None:
                continue

            data = obj.get("data", {}) or {}
            frames.append(
                {
                    "step": step_idx,
                    "timestamp": obj.get("timestamp"),
                    "game_id": data.get("game_id"),
                    "state": data.get("state"),
                    "levels_completed": data.get("levels_completed", 0),
                    "win_levels": data.get("win_levels", 0),
                    "grid": normalize_frame_to_single_grid(data.get("frame")),
                }
            )
    return frames


def build_lightweight_index(working_dir: Path) -> dict[str, dict[str, str]]:
    """Build a lightweight index of available recordings.

    Index structure:
    run_name -> game_key -> jsonl_file_path

    This function intentionally avoids parsing recording content for speed.
    """
    db = {}

    run_dirs = sorted(
        [p for p in working_dir.iterdir() if p.is_dir() and p.name.startswith("runs-")],
        key=lambda p: p.name,
        reverse=True,
    )

    for run_dir in run_dirs:
        recordings_dir = run_dir / "recordings"
        if not recordings_dir.exists() or not recordings_dir.is_dir():
            continue

        games = {}
        for fp in sorted(recordings_dir.glob("*.jsonl")):
            game_key = fp.name.split(".", 1)[0]
            games[game_key] = str(fp)

        if games:
            db[run_dir.name] = games

    return db


def grid_to_base64_png(grid, canvas_size: int = CANVAS_SIZE) -> str | None:
    """Render one ARC grid as a centered PNG image and return base64 text."""
    if grid is None:
        return None

    arr = np.array(grid, dtype=int)
    if arr.ndim != 2 or arr.size == 0:
        return None

    # Clamp invalid color indices to 0 for robust rendering.
    arr = np.where((0 <= arr) & (arr < len(ARC_COLORS_RGB)), arr, 0)
    rgb = ARC_COLORS_RGB[arr]

    rows, cols = arr.shape
    base = Image.fromarray(rgb, mode="RGB")

    scale = max(1, canvas_size // max(rows, cols))
    draw_w = cols * scale
    draw_h = rows * scale

    scaled = base.resize((draw_w, draw_h), Image.Resampling.NEAREST)

    canvas = Image.new("RGB", (canvas_size, canvas_size), (0, 0, 0))
    offset_x = (canvas_size - draw_w) // 2
    offset_y = (canvas_size - draw_h) // 2
    canvas.paste(scaled, (offset_x, offset_y))

    buf = io.BytesIO()
    canvas.save(buf, format="PNG", compress_level=1)
    return base64.b64encode(buf.getvalue()).decode("ascii")


def make_board_html(img_b64: str | None, message: str | None = None) -> str:
    """Return HTML for either an image board or a text placeholder."""
    if img_b64 is None:
        text = message or "Select a run and game, then click Render."
        return f"""
        <div class="arc-board-shell">
            <div class="arc-empty">{text}</div>
        </div>
        """

    return f"""
    <div class="arc-board-shell">
        <img class="arc-board-img" src="data:image/png;base64,{img_b64}" />
    </div>
    """


def make_meta_html(step=None, levels_completed=None, win_levels=None) -> str:
    """Return top status HTML for the currently selected frame."""
    if step is None:
        return """
        <div class="arc-meta-row">
            <div>STEP: -</div>
            <div>LEVELS COMPLETED: -</div>
        </div>
        """

    return f"""
    <div class="arc-meta-row">
        <div>STEP: {step}</div>
        <div>LEVELS COMPLETED: {levels_completed} / {win_levels}</div>
    </div>
    """


# ---------- CSS ----------
display(
    HTML(
        """
<style>
.arc-wrap {
    width: 75%;
    margin: 0 auto;
    background: #111111;
    border: 1px solid #252525;
    border-radius: 8px;
    padding: 18px;
    box-sizing: border-box;
    color: #E8E8E8;
    font-family: sans-serif;
}
.arc-controls {
    margin-bottom: 14px;
}
.arc-status {
    margin-bottom: 14px;
}
.arc-board {
    margin-bottom: 16px;
}
.arc-playback {
    margin-top: 16px;
}
.arc-meta-row {
    display: flex;
    justify-content: center;
    align-items: center;
    gap: 24px;
    flex-wrap: wrap;
    color: #D8D8D8;
    font-weight: 700;
    font-size: 14px;
}
.arc-board-shell {
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 580px;
}
.arc-board-img {
    width: 100%;
    max-width: 560px;
    aspect-ratio: 1 / 1;
    object-fit: contain;
    display: block;
    background: #000000;
    border: 1px solid #2E2E2E;
    box-sizing: border-box;
    image-rendering: pixelated;
}
.arc-empty {
    color: #8A8A8A;
    font-size: 14px;
    text-align: center;
}
.arc-wrap select,
.arc-wrap button,
.arc-wrap input[type="range"] {
    font-family: sans-serif;
}
</style>
"""
    )
)

# ---------- Build an index only (no heavy JSONL parsing yet) ----------
runs_index = build_lightweight_index(WORKING_DIR)

if not runs_index:
    display(
        HTML(
            """
    <div class="arc-wrap">
        No <code>runs-*</code> directories were found in <code>/kaggle/working</code>.
    </div>
    """
        )
    )
else:
    # ---------- Runtime state ----------
    current_frames = []
    current_render_cache = {}

    # ---------- Widgets ----------
    run_dropdown = widgets.Dropdown(
        options=list(runs_index.keys()),
        description="",
        layout=widgets.Layout(width="300px"),
    )

    game_dropdown = widgets.Dropdown(
        options=[],
        description="",
        layout=widgets.Layout(width="240px"),
    )

    render_button = widgets.Button(
        description="Render",
        button_style="primary",
        layout=widgets.Layout(width="110px"),
    )

    play_widget = widgets.Play(
        value=0,
        min=0,
        max=0,
        step=1,
        interval=int(1000 / FPS),
        description="",
        disabled=True,
    )

    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=0,
        step=1,
        readout=False,
        continuous_update=False,
        disabled=True,
        layout=widgets.Layout(flex="1 1 auto", width="auto"),
    )

    # Keep slider and play widget in sync.
    widgets.jslink((play_widget, "value"), (slider, "value"))

    frame_count_html = widgets.HTML(value='<span style="font-size:12px;color:#8E8E8E;">0 / 0</span>')

    status_html = widgets.HTML(value=make_meta_html())

    board_html = widgets.HTML(value=make_board_html(None, "Select a run and game, then click Render."))

    info_html = widgets.HTML(
        value='<span style="font-size:12px;color:#8E8E8E;">No game has been rendered yet.</span>'
    )

    def update_game_dropdown(*_) -> None:
        """Refresh game options when the selected run changes."""
        run_name = run_dropdown.value
        games = sorted(runs_index.get(run_name, {}).keys())
        game_dropdown.options = games
        if games:
            game_dropdown.value = games[0]

    update_game_dropdown()
    run_dropdown.observe(update_game_dropdown, names="value")

    def render_frame(idx: int) -> None:
        """Render one frame by index and update status/footer widgets."""
        if not current_frames:
            status_html.value = make_meta_html()
            board_html.value = make_board_html(None, "No frames.")
            frame_count_html.value = '<span style="font-size:12px;color:#8E8E8E;">0 / 0</span>'
            return

        idx = max(0, min(idx, len(current_frames) - 1))
        frame = current_frames[idx]

        status_html.value = make_meta_html(
            step=frame["step"],
            levels_completed=frame["levels_completed"],
            win_levels=frame["win_levels"],
        )

        # Cache generated image HTML so scrubbing is smoother.
        if idx not in current_render_cache:
            img_b64 = grid_to_base64_png(frame["grid"])
            current_render_cache[idx] = make_board_html(img_b64, "No board.")
        board_html.value = current_render_cache[idx]

        frame_count_html.value = (
            f'<span style="font-size:12px;color:#8E8E8E;">{idx + 1} / {len(current_frames)}</span>'
        )

    def on_render_clicked(_) -> None:
        """Load the selected recording file and reset playback state."""
        selected_run = run_dropdown.value
        selected_game = game_dropdown.value

        if not selected_run or not selected_game:
            return

        path = Path(runs_index[selected_run][selected_game])

        render_button.disabled = True
        render_button.description = "Loading..."

        try:
            frames = load_recording_jsonl(path)

            current_frames.clear()
            current_frames.extend(frames)
            current_render_cache.clear()

            max_idx = max(0, len(current_frames) - 1)
            slider.max = max_idx
            slider.value = 0
            slider.disabled = len(current_frames) == 0

            play_widget.max = max_idx
            play_widget.value = 0
            play_widget.disabled = len(current_frames) == 0

            info_html.value = (
                f'<span style="font-size:12px;color:#8E8E8E;">'
                f"{selected_run} | {selected_game} | {len(current_frames)} frames"
                f"</span>"
            )

            render_frame(0)

        except Exception as e:
            current_frames.clear()
            current_render_cache.clear()
            slider.max = 0
            slider.value = 0
            slider.disabled = True
            play_widget.max = 0
            play_widget.value = 0
            play_widget.disabled = True

            status_html.value = make_meta_html()
            board_html.value = make_board_html(None, f"Failed to load file: {e}")
            frame_count_html.value = '<span style="font-size:12px;color:#8E8E8E;">0 / 0</span>'
            info_html.value = '<span style="font-size:12px;color:#D66;">Load failed.</span>'

        finally:
            render_button.disabled = False
            render_button.description = "Render"

    render_button.on_click(on_render_clicked)

    def on_slider_change(change) -> None:
        """Render the current frame whenever slider/playback position changes."""
        if change["name"] == "value" and change["new"] is not None and current_frames:
            render_frame(change["new"])

    slider.observe(on_slider_change, names="value")

    # ---------- Layout ----------
    controls_row = widgets.HBox(
        [run_dropdown, game_dropdown, render_button],
        layout=widgets.Layout(gap="10px", flex_flow="row wrap", align_items="center"),
    )

    playback_row = widgets.HBox(
        [play_widget, slider, frame_count_html],
        layout=widgets.Layout(gap="12px", align_items="center"),
    )

    controls_box = widgets.VBox(
        [controls_row, info_html],
        layout=widgets.Layout(padding="12px", border="1px solid #242424", margin="0 0 14px 0"),
    )
    controls_box.add_class("arc-controls")

    status_box = widgets.VBox([status_html], layout=widgets.Layout(margin="0 0 14px 0"))
    status_box.add_class("arc-status")

    board_box = widgets.VBox([board_html], layout=widgets.Layout(margin="0 0 16px 0"))
    board_box.add_class("arc-board")

    playback_box = widgets.VBox(
        [playback_row],
        layout=widgets.Layout(padding="12px", border="1px solid #242424"),
    )
    playback_box.add_class("arc-playback")

    outer = widgets.VBox([controls_box, status_box, board_box, playback_box])
    outer.add_class("arc-wrap")

    display(outer)

## Cell 6 - Scratch / Follow-up Analysis

A free workspace for additional checks after rendering:
- inspect result paths,
- compare scorecards between runs,
- prototype custom diagnostics without changing core cells.